In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm, trange

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torchvision.transforms import ToTensor
from torchvision import models
from torch.utils.data import WeightedRandomSampler

np.random.seed(0)
torch.manual_seed(0)

In [ ]:
# data augmentation pipeline
import torchvision.transforms as transforms
from torchvision import datasets

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(30),
    transforms.Resize((224, 224))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224))
])

train_set = datasets.ImageFolder(root = 'Splitted2D/train', transform = train_transform)
val_set = datasets.ImageFolder(root = 'Splitted2D/val', transform = val_transform)
test_set = datasets.ImageFolder(root = 'Splitted2D/test', transform = test_transform)

In [ ]:
idx2class = {v: k for k, v in train_set.class_to_idx.items()}

def get_class_distribution(dataset_obj):
    count_dict = {k:0 for k,v in dataset_obj.class_to_idx.items()}
    
    for element in dataset_obj:
        y_lbl = element[1]
        y_lbl = idx2class[y_lbl]
        count_dict[y_lbl] += 1
            
    return count_dict
print("Distribution of classes: \n", get_class_distribution(train_set))

In [ ]:
BATCH_SIZE = 32

def get_class_distribution_loaders(dataloader_obj, dataset_obj):
    count_dict = {k:0 for k,v in dataset_obj.class_to_idx.items()}
    
    for _,j in dataloader_obj:
        y_idx = j.item()
        y_lbl = idx2class[y_idx]
        count_dict[str(y_lbl)] += 1
            
    return count_dict

target_list = torch.tensor(train_set.targets)
class_count = [i for i in get_class_distribution(train_set).values()]
class_weights = 1./torch.tensor(class_count, dtype=torch.float) 
print(class_weights)
###################### OUTPUT ######################
#tensor([0.0068, 0.0054, 0.0114, 0.0123])

class_weights_all = class_weights[target_list]

weighted_sampler = WeightedRandomSampler(
    weights=class_weights_all,
    num_samples=len(class_weights_all),
    replacement=False
)

train_loader = DataLoader(dataset = train_set, shuffle = False, batch_size = BATCH_SIZE, sampler = weighted_sampler)
val_loader = DataLoader(dataset = val_set, shuffle = True, batch_size = BATCH_SIZE)
test_loader = DataLoader(dataset = test_set, shuffle = True, batch_size = BATCH_SIZE)

In [ ]:
model = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size = (3, 3), stride = (1, 1), padding = (1, 1)),
    nn.BatchNorm2d()
)